In [1]:
!pip install -q langchain langchain-community langchain-ollama langgraph chromadb rdflib sentence-transformers rank_bm25 numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.

In [2]:
# System dependency needed to extract the Ollama binary
!apt-get update -qq && apt-get install -y -qq zstd

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time

print("\nPornim serverul Ollama...")
subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(3)

# mistral-nemo supports tool calling — llama3 (base) does NOT, don't switch back
print("Descarcam modelul mistral-nemo (te rog asteapta, ~2-3 min)...")
!ollama pull mistral-nemo

print(" Ollama & mistral-nemo sunt instalate și rulează LOCAL!")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

Pornim serverul Ollama...
Descarcam modelul mistral-nemo (te rog asteapta, ~2-

In [3]:
import numpy as np
from sentence_transformers import SentenceTransformer
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.retrievers import BM25Retriever
from langchain.tools import tool


CHUNK_SIZE = 500
CHUNK_OVERLAP = 75
TOP_K = 4
EMBEDDING_MODEL = "all-MiniLM-L6-v2"


try:
    loader = TextLoader("lore.txt", encoding="utf-8")
    docs = loader.load()
    print("Unstructured lore loaded successfully.")
except Exception as error:
    print(f"Error loading lore.txt: {error}. Upload the file before continuing.")
    docs = []


if docs:
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )
    splits = text_splitter.split_documents(docs)

    for chunk_id, document in enumerate(splits):
        document.metadata["chunk_id"] = chunk_id

    # Dense semantic index: normalized vectors make the dot product equivalent
    # to cosine similarity.
    embedding_model = SentenceTransformer(EMBEDDING_MODEL)
    chunk_embeddings = embedding_model.encode(
        [document.page_content for document in splits],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype("float32")

    # Lexical index: BM25 complements embeddings with exact-name and keyword matches.
    bm25_retriever = BM25Retriever.from_documents(splits)


    def hybrid_retrieve(query: str, top_k: int = TOP_K, candidate_k: int = 12):
        """Combine dense semantic and BM25 rankings with Reciprocal Rank Fusion."""
        candidate_k = min(candidate_k, len(splits))

        query_embedding = embedding_model.encode(
            [query],
            convert_to_numpy=True,
            normalize_embeddings=True,
        )[0].astype("float32")

        dense_scores = chunk_embeddings @ query_embedding
        dense_ids = np.argsort(dense_scores)[::-1][:candidate_k].tolist()

        bm25_retriever.k = candidate_k
        lexical_docs = bm25_retriever.invoke(query)
        lexical_ids = [document.metadata["chunk_id"] for document in lexical_docs]

        fused_scores = {}
        rrf_constant = 60

        for rank, chunk_id in enumerate(dense_ids, start=1):
            fused_scores[chunk_id] = fused_scores.get(chunk_id, 0.0) + 1 / (
                rrf_constant + rank
            )

        for rank, chunk_id in enumerate(lexical_ids, start=1):
            fused_scores[chunk_id] = fused_scores.get(chunk_id, 0.0) + 1 / (
                rrf_constant + rank
            )

        ranked_ids = sorted(
            fused_scores,
            key=fused_scores.get,
            reverse=True,
        )[:top_k]

        return [splits[chunk_id] for chunk_id in ranked_ids]


    @tool
    def search_lore(query: str) -> str:
        """Search narrative lore and biographies with hybrid embedding and BM25 retrieval."""
        results = hybrid_retrieve(query)
        if not results:
            return "No relevant lore was found."

        return "\n\n".join(
            f"[lore:{document.metadata['chunk_id']}] {document.page_content}"
            for document in results
        )


    rag_tool = search_lore
    print(
        f"Hybrid RAG is ready: {len(splits)} chunks embedded with "
        f"{EMBEDDING_MODEL} and indexed with BM25."
    )


/tmp/ipykernel_566/4004361784.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Unstructured lore loaded successfully.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Hybrid RAG is ready: 739 chunks embedded with all-MiniLM-L6-v2 and indexed with BM25.


In [4]:
from langchain.tools import tool
import rdflib

@tool
def query_graph_db(character_name: str) -> str:
    """
    Use this tool to look up structured facts about a character: who they killed,
    who killed them, their family (siblings, parents, children), who they serve,
    or who is married to them. Pass the character's name.
    """
    try:
        g = rdflib.ConjunctiveGraph()
        g.parse("graph.trig", format="trig")

        # Graph subjects use underscores instead of spaces (e.g. Ned_Stark)
        normalized_name = character_name.strip().replace(" ", "_")

        query = f"""
        SELECT ?predicate ?object
        WHERE {{
            ?subject ?predicate ?object .
            FILTER(CONTAINS(STR(?subject), "{normalized_name}"))
        }}
        LIMIT 10
        """

        results = g.query(query)
        output_lines = []

        for row in results:
            pred = str(row.predicate).split('/')[-1].split('#')[-1]
            obj = str(row.object).split('/')[-1].split('#')[-1].replace('_', ' ')
            output_lines.append(f"- {pred}: {obj}")

        if output_lines:
            return f"Structured Graph Data for {character_name}:\n" + "\n".join(output_lines)
        else:
            return f"No structured data found in the graph for {character_name}."

    except Exception as e:
        return f"Error reading graph.trig: {e}. Make sure the file exists and is formatted correctly."

print(" Knowledge Graph Tool (from TriG) has been configured!")

 Knowledge Graph Tool (from TriG) has been configured!


In [5]:
from langchain_ollama import ChatOllama
from langgraph.prebuilt import create_react_agent


tools = [query_graph_db]
if "rag_tool" in locals():
    tools.append(rag_tool)

# Bound answer length and keep the model loaded between questions. This does not
# change the model; it prevents unnecessarily long local generations.
llm = ChatOllama(
    model="mistral-nemo",
    temperature=0,
    num_predict=128,
    num_ctx=4096,
    keep_alive="30m",
)

system_prompt = """You are a grounded Game of Thrones retrieval assistant.

Use search_lore for narrative text, biographies, aliases, titles, culture and appearances.
Use query_graph_db for kills, family, service and marriage.
Use both tools only when the question genuinely requires both sources.

Call each required tool at most once. Never retry a tool that returned no relevant data.
Never use pretrained Game of Thrones knowledge; answer only from tool results.
If the retrieved evidence does not answer the question, say:
I do not have this information in the provided context.

Keep the final answer direct and concise.
"""

agent_executor = create_react_agent(llm, tools, prompt=system_prompt)
print("The local agent is ready.")


The local agent is ready.


/tmp/ipykernel_566/277197147.py:33: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(llm, tools, prompt=system_prompt)


In [ ]:
import subprocess
import time
import requests

from langchain_core.messages import AIMessage, HumanMessage, ToolMessage


def ensure_ollama_running():
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2).raise_for_status()
    except requests.RequestException:
        subprocess.Popen(
            ["ollama", "serve"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        time.sleep(5)


def message_text(content) -> str:
    """Normalize either string or block-style model content."""
    if isinstance(content, str):
        return content.strip()
    if isinstance(content, list):
        return "".join(
            block.get("text", "")
            for block in content
            if isinstance(block, dict)
        ).strip()
    return ""


ensure_ollama_running()
print("Welcome to the Network of Thrones Assistant. Type 'exit' or 'quit' to stop.")

while True:
    user_input = input("\nUser: ").strip()

    if not user_input:
        continue
    if user_input.casefold() in {"exit", "quit"}:
        print("System shutting down. Valar Morghulis!")
        break

    started = time.perf_counter()

    try:
        result = agent_executor.invoke(
            {"messages": [HumanMessage(content=user_input)]},
            config={"recursion_limit": 6},
        )

        for message in result.get("messages", []):
            if isinstance(message, AIMessage):
                for call in message.tool_calls or []:
                    print(f"Tool called: {call.get('name', 'unknown_tool')}")
            elif isinstance(message, ToolMessage):
                print(f"Retrieved:\n{str(message.content).strip()}")

        final_response = ""
        for message in reversed(result.get("messages", [])):
            if isinstance(message, AIMessage) and not message.tool_calls:
                final_response = message_text(message.content)
                if final_response:
                    break

        if not final_response:
            final_response = "I do not have this information in the provided context."

        elapsed = time.perf_counter() - started
        print(f"\nAssistant: {final_response}")
        print(f"Response time: {elapsed:.1f} seconds")

    except Exception as error:
        print(f"Request failed: {type(error).__name__}: {error}")


Welcome to the Network of Thrones Assistant. Type 'exit' or 'quit' to stop.

User: Who did Arya Stark kill?


/tmp/ipykernel_566/1567315349.py:12: DeprecationWarning: ConjunctiveGraph is deprecated, use Dataset instead.
  g = rdflib.ConjunctiveGraph()


Tool called: query_graph_db
Retrieved:
Structured Graph Data for Arya Stark:
- childOf: Catelyn Stark
- killed: The Waif
- killed: Petyr Baelish
- killed: Viserion
- killed: Black Walder Rivers
- siblingOf: Bran Stark
- killed: Ghita
- siblingOf: Robb Stark
- killed: Rorge
- killed: Red Keep Stableboy

Assistant: Arya Stark has killed several characters throughout the series. Here's a list of them:

1. **The Waif**: Arya kills her in Season 6, Episode 9 ("Battle of the Bastards") as part of her revenge plan against the Faceless Men.
2. **Petyr Baelish (Littlefinger)**: Arya stabs him to death in Season 7, Episode 7 ("The Dragon and the Wolf"), fulfilling a promise she made to Sansa Stark.
3. **Viserion**: Arya shoots Viserion with a dragon-killing spear provided by Daenerys T
Response time: 376.9 seconds

User: What is the backstory of Eddard Stark?
